End-to-End Example for Embedding Creation Using Agent Server Types

This example demonstrates how to:
1. Set up a Cortex platform client
2. Create embeddings using direct Model selection
3. Create embeddings using ModelSelector (for dynamic model selection)
4. Compare the outputs and understand the embedding response format

To run this example:
1. Setup proper AWS credentials in your environment (AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY)

In [1]:
from utils import MockKernel, setup_notebook

if not setup_notebook():
    raise ValueError("Failed to setup notebook")

In [2]:
import pprint
from typing import Any, cast

import numpy as np

from agent_platform.core.kernel import Kernel
from agent_platform.core.model_selector import (
    DefaultModelSelector,
    ModelSelectionRequest,
)
from agent_platform.core.platforms.cortex.client import CortexClient
from agent_platform.core.platforms.cortex.configs import CortexModelMap
from agent_platform.core.platforms.cortex.parameters import CortexPlatformParameters

pp = pprint.PrettyPrinter(indent=4)

In [3]:
async def create_cortex_client() -> CortexClient:
    """Create and initialize a Cortex client.

    Returns:
        An initialized CortexClient
    """
    # Create the client with AWS credentials
    client = CortexClient(parameters=CortexPlatformParameters())
    client.attach_kernel(cast(Kernel, MockKernel()))

    print("✅ Cortex client created successfully")
    return client

In [4]:
async def run_direct_model_embeddings(
    client: CortexClient,
    texts: list[str],
) -> dict[str, Any]:
    """Create embeddings using a direct Model reference.

    Args:
        client: The CortexClient instance
        texts: List of text strings to embed

    Returns:
        The embedding response
    """
    print("\n=== Creating Embeddings with Direct Model Reference ===")

    # Select a specific embedding model from our Models catalog
    model = "snowflake-arctic-embed-m"
    full_name = CortexModelMap[model]
    print(f"Using model: {model} ({full_name})")

    # Create embeddings
    result = await client.create_embeddings(
        texts=texts,
        model=model,
    )

    return result


async def run_model_selector_embeddings(
    client: CortexClient,
    texts: list[str],
) -> dict[str, Any]:
    """Create embeddings using a ModelSelector for dynamic model selection.

    Args:
        client: The CortexClient instance
        texts: List of text strings to embed

    Returns:
        The embedding response
    """
    print("\n=== Creating Embeddings with ModelSelector ===")

    # Create a model selector with a preference list
    # In a real application, this might include complex selection logic
    model_selector = DefaultModelSelector()

    # The actual model will be selected by the ModelSelector at runtime
    print("Using ModelSelector (will select appropriate model dynamically)")

    # Create embeddings using the model selector
    result = await client.create_embeddings(
        texts=texts,
        model=model_selector.select_model(
            client,
            ModelSelectionRequest(
                model_type="embedding",
                quality_tier="balanced",
            ),
        ),
    )

    # In the actual implementation, the client calls model_selector.select_model()
    # internally to resolve the model_selector to a concrete Model

    return result

In [5]:
def analyze_embedding_results(
    direct_model_result: dict[str, Any],
    selector_result: dict[str, Any],
) -> None:
    """Analyze and compare embedding results from both approaches.

    Args:
        direct_model_result: Result from direct model embeddings
        selector_result: Result from model selector embeddings
    """
    print("\n=== Analyzing Embedding Results ===")

    # Analyze direct model result
    print("\nDirect Model Result:")
    print(f"Model used: {direct_model_result['model']}")
    print(f"Number of embeddings: {len(direct_model_result['embeddings'])}")
    print(f"Embedding dimension: {len(direct_model_result['embeddings'][0])}")
    print(f"Total tokens used: {direct_model_result['usage']['total_tokens']}")

    # Analyze selector result
    print("\nModelSelector Result:")
    print(f"Model used: {selector_result['model']}")
    print(f"Number of embeddings: {len(selector_result['embeddings'])}")
    print(f"Embedding dimension: {len(selector_result['embeddings'][0])}")
    print(f"Total tokens used: {selector_result['usage']['total_tokens']}")

    # Check if results are similar (they should be if the same model was selected)
    models_match = direct_model_result["model"] == selector_result["model"]
    dimensions_match = len(direct_model_result["embeddings"][0]) == len(
        selector_result["embeddings"][0],
    )
    # Check if the embeddings are similar
    # Calculate the cosine similarity between the embeddings
    similarity = np.dot(
        direct_model_result["embeddings"][0],
        selector_result["embeddings"][0],
    ) / (
        np.linalg.norm(direct_model_result["embeddings"][0])
        * np.linalg.norm(selector_result["embeddings"][0])
    )
    print(f"Cosine similarity between embeddings: {similarity}")

    print("\nComparison:")
    print(f"Same model used: {'✅' if models_match else '❌'}")
    print(f"Same embedding dimensions: {'✅' if dimensions_match else '❌'}")
    print(f"Cosine similarity between embeddings: {similarity}")

In [6]:
async def run_e2e_embeddings():
    """Main function to run the example."""
    print("Starting Embeddings E2E Example...")

    # Sample texts to embed
    sample_texts = [
        "Artificial intelligence is transforming how we work and live.",
        "Machine learning algorithms find patterns in large datasets.",
        "Natural language processing helps computers understand human language.",
    ]

    # Create Cortex client
    client = await create_cortex_client()

    # Create embeddings using both approaches
    direct_result = await run_direct_model_embeddings(client, sample_texts)
    selector_result = await run_model_selector_embeddings(client, sample_texts)

    # Print a sample of the embedding vectors (just first 5 dimensions)
    print("\n=== Sample of Embedding Vectors (first 5 dimensions) ===")
    print("\nDirect Model - First text embedding:")
    pp.pprint(direct_result["embeddings"][0][:5])

    print("\nModelSelector - First text embedding:")
    pp.pprint(selector_result["embeddings"][0][:5])

    # Analyze results
    analyze_embedding_results(direct_result, selector_result)

    print("\n✅ E2E Embeddings Example Complete")

In [7]:
await run_e2e_embeddings()

Starting Embeddings E2E Example...
✅ Cortex client created successfully

=== Creating Embeddings with Direct Model Reference ===
Using model: snowflake-arctic-embed-m (snowflake-arctic-embed-m-v1.5)

=== Creating Embeddings with ModelSelector ===
Using ModelSelector (will select appropriate model dynamically)

=== Sample of Embedding Vectors (first 5 dimensions) ===

Direct Model - First text embedding:
[   0.0867919921875,
    -0.02203369140625,
    0.1055908203125,
    0.01432037353515625,
    0.08551025390625]

ModelSelector - First text embedding:
[   0.0867919921875,
    -0.02203369140625,
    0.1055908203125,
    0.01432037353515625,
    0.08551025390625]

=== Analyzing Embedding Results ===

Direct Model Result:
Model used: snowflake-arctic-embed-m
Number of embeddings: 3
Embedding dimension: 768
Total tokens used: 47

ModelSelector Result:
Model used: snowflake-arctic-embed-m
Number of embeddings: 3
Embedding dimension: 768
Total tokens used: 47
Cosine similarity between embedd